# Step 7 — Zero-Leakage 5-Fold Downstream Fine-Tuning (`LP-FT` vs `Scratch`) & Integrated Gradients XAI

This interactive HPC notebook lets you evaluate **any pre-trained encoder (`1D-ViT SMAE` or `1D-ResNet-34 Barlow Twins`)** on **any of the 4 downstream benchmarks** using our mathematically verified **0% patient/subject leakage 5-fold splits (`data/02_ready/downstream/splits/`)**:
- **`configs/finetune/smae_hw_on_riverd_oral.yaml`**: Pre-trained `ViT-SMAE (HW)` $\rightarrow$ **`riverd_oral_hw`** (`8,422` spectra, `186` surgical patients, Normal vs OSCC Tumor)
- **`configs/finetune/barlow_twins_on_riverd_oral.yaml`**: Pre-trained `Barlow Twins (ResNet-34)` $\rightarrow$ **`riverd_oral_hw`**
- **`configs/finetune/barlow_twins_on_ibd.yaml`**: Pre-trained `Barlow Twins (ResNet-34)` $\rightarrow$ **`ibd`** (`228` clinical plasma subjects)
- **`configs/finetune/barlow_twins_on_bacteria.yaml`**: Pre-trained `Barlow Twins (ResNet-34)` $\rightarrow$ **`bacteria_id`** (`30` pathogenic species, `3k` finetune + `3k` clinical test)
- **`configs/finetune/barlow_twins_on_renzo.yaml`**: Pre-trained `Barlow Twins (ResNet-34)` $\rightarrow$ **`dataset_renzo`** (`3,200` spectra, `47` mice, `4` pathologies)

> **Note**: Upload the lightweight `data/02_ready/downstream/` folder (`~207 MB` total) to your HPC workspace before running.

## 1. Select Preset Configuration & Optional Overrides
Change `CONFIG_FILE` (or set any override like `CHECKPOINT_OVERRIDE` / `OUT_DIR_OVERRIDE`) to choose which model and benchmark to evaluate.

In [ ]:
from pathlib import Path
import torch, yaml, pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
CONFIG_FILE = 'configs/finetune/smae_hw_on_riverd_oral.yaml'  # <--- Pick any preset here!
CHECKPOINT_OVERRIDE = None  # e.g. 'experiments/smae_hw_exp1_p100_m50/best_smae_model.pth'
OUT_DIR_OVERRIDE = None     # e.g. 'experiments/ft_smae_hw_riverd_exp2'
RUN_SCRATCH = True          # Compare against random-init Scratch Baseline on identical folds

print('Device:', 'cuda (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'cpu')
print('Selected config:', CONFIG_FILE)

## 2. Inspect Selected Fine-Tuning Configuration
Review LP-FT learning rates, BatchNorm freezing, augmentation settings, and checkpoint path.

In [ ]:
cfg = yaml.safe_load((ROOT / CONFIG_FILE).read_text(encoding='utf-8'))
print(yaml.safe_dump(cfg, sort_keys=False))

## 3. Verify Downstream Dataset & Zero-Leakage Fold 0 Split
Confirm that `data/02_ready/downstream/` and `splits/` are present and have `0` patient/subject overlap.

In [ ]:
from spectrabreast.data import DownstreamFoldLoader

loader = DownstreamFoldLoader(ROOT / 'data' / '02_ready')
f0 = loader.get_fold(cfg['data']['benchmark'], fold=0, regime=cfg['data'].get('regime', 'default'))
ov = len(set(f0.get('train_groups', [])).intersection(set(f0.get('test_groups', []))))
print(f"Fold 0 | Train: {f0['X_train'].shape} | Test: {f0['X_test'].shape} | Subject Overlap: {ov}")

## 4. Run Full 5-Fold Zero-Leakage Fine-Tuning (`LP-FT` vs `Scratch`) + Integrated Gradients XAI
Runs all 5 folds, logs per-spectrum and per-subject metrics, generates all 300-DPI plots, and saves `notebook_eseguito.ipynb` inside the experiment folder.

In [ ]:
from spectrabreast.training import run_downstream_finetuning

res = run_downstream_finetuning(
    config_path=ROOT / CONFIG_FILE,
    repo_root=ROOT,
    checkpoint_override=CHECKPOINT_OVERRIDE,
    output_dir_override=OUT_DIR_OVERRIDE,
    run_scratch_override=RUN_SCRATCH,
)
display(res['results_df'])

## 5. Transfer Learning Gain: 5-Fold `Scratch` vs `Pre-Trained LP-FT`
Inspect Accuracy, Balanced Accuracy, Macro-F1, and ROC-AUC across all 5 zero-leakage folds.

In [ ]:
fig_dir = res['output_dir'] / 'figures'
display(Image(filename=str(fig_dir / 'ft_scratch_vs_pretrained_comparison.png')))

## 6. LP-FT Optimization Curves (`Start FT` Transition) & Clinical Evaluation (Confusion Matrix / ROC / PR)
Inspect Stage 1 (Linear Probing) $\rightarrow$ Stage 2 (Fine-Tuning) convergence and aggregated out-of-fold clinical metrics.

In [ ]:
display(Image(filename=str(fig_dir / 'ft_learning_curves_fold1.png')))
display(Image(filename=str(fig_dir / 'ft_clinical_evaluation.png')))
prob_fig = fig_dir / 'ft_probability_distribution.png'
if prob_fig.exists():
    display(Image(filename=str(prob_fig)))

## 7. Explainable AI (XAI) — Pure-PyTorch Integrated Gradients & Top Discriminative Raman Bands
Inspect which physical Raman shift regions ($\text{cm}^{-1}$) drove the fine-tuned classifier's clinical predictions.

In [ ]:
ig_fig = fig_dir / 'ft_integrated_gradients_xai.png'
peaks_csv = res['output_dir'] / 'top_peaks_ig.csv'
if ig_fig.exists():
    display(Image(filename=str(ig_fig)))
if peaks_csv.exists():
    display(pd.read_csv(peaks_csv))